In [2]:
import os 
os.chdir('../../')

!nvidia-smi

Mon Sep  1 00:16:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.76.05              Driver Version: 580.76.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:19:00.0 Off |                  N/A |
| 30%   49C    P8             13W /  575W |      41MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from backbones.sana import SANA

# 사용 예
model = SANA(model_id='Efficient-Large-Model/Sana_600M_512px_diffusers')
print(model)

/home/scpark/miniconda3/envs/dual/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-01 00:16:34.921504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  3.05it/s]


In [4]:
!ls logs/sana/0822-14:SANA,3steps,CLIP,clip_loss,scheduling,tau=1

events.out.tfevents.1756649791.scpark-X299-WU8.23280.0
events.out.tfevents.1756650214.scpark-X299-WU8.25205.0
events.out.tfevents.1756651060.scpark-X299-WU8.28598.0
events.out.tfevents.1756651107.scpark-X299-WU8.28936.0


In [5]:
# 1) Euler 20steps, 2) Euler 5steps, 3) DPM 5steps, 4) GDual 5steps — PIL 가로 비교 (compact)
import torch, matplotlib.pyplot as plt
from PIL import Image
from solvers.others.euler_solver import Euler_Solver
from solvers.others.dpm_solver import DPM_Solver
from solvers.taylor.solver.gdual_solver import GDual_Solver
from solvers.taylor.transform.logaffine_transform import LogAffineTransform
from solvers.taylor.extractor.table_extractor import Extractor

steps = 3
flow_shift = 1.0
#pt_file = '/data/scpark/logs/sana/0819-2:SANA,CLIP,4steps/step_00001500.pt'  # GDual checkpoint
#pt_file = '/data/scpark/logs/sana/0820-1:SANA,CLIP,6steps,batch/step_00010500.pt'  # GDual checkpoint
pt_file = 'logs/sana/0821-16:SANA,OpenCLIP,6steps,inception/step_00001400.pt'

pos_texts = [
    "Rain-soaked night market in Busan, neon reflections on wet asphalt, steaming tteokbokki stalls, colorful umbrellas, candid people in motion; cinematic 8K photorealism, 35mm f/1.4, shallow depth, moody teal-orange grade, soft bokeh, natural grain",
    #"Glacial valley under vivid aurora, mirrorlike fjord, drifting ice floes, lone wooden cabin; ultra-wide 14mm long exposure, crisp stars, cold airy palette, serene atmosphere, high dynamic range, subtle fog",
    #"Minimal studio shot of a matte-black ceramic teapot set with steam curls, walnut tray, linen backdrop; clean product render, ray-traced reflections, softbox lighting, hard edges razor-sharp, dust-free, premium mood",
    #"Jumping spider on moss with dewdrops, iridescent eyes, fine hair detail; 100mm macro f/2.8, focus stack look, ring-flash speculars, crisp micro-contrast, clinical clarity, neutral colorimetry",
    "Kelp forest with sunbeams and dappled caustics, playful sea otter and small fish schools, floating particles; 16mm dome-port aesthetic, cool cyan-green palette, tranquil, high clarity, volumetric light",
    #"Brutalist museum atrium, concrete ribs, suspended staircase, lone bronze sculpture; 24mm tilt-shift, straight verticals, mid-day sunbeam through skylight, minimal visitors, spacious and contemplative",
    "Ink-wash painting of a dragon coiling around a mountain pagoda, swirling clouds and pine silhouettes; sumi-e brushwork, rice paper texture, restrained grayscale with single red seal, elegant and timeless",
    "Cel-shaded anime alley at blue hour, holographic billboards, vending machines, rain haze, scooter passing; clean lineart, crisp highlights, halftone shadows, punchy magenta-cyan palette, upbeat yet gritty",
    #"Isometric voxel mountain village at night: tiny pine trees, tiled roofs, a red commuter train crossing a bridge, warm window lights; playful, clean ambient occlusion, miniature tilt-shift look",
    #"Editorial portrait of a jazz saxophonist on a stool, swirling smoke, vintage microphone; Rembrandt lighting, 85mm f/1.8, low-key background, natural skin texture, warm amber highlights, intimate and soulful",
]
for pos in pos_texts:
    neg = ["lowres, bad anatomy, deformed, blurry, pixelated, oversaturated, underexposed, overexposed, artifact, jpeg artifacts, watermark, text, logo, extra limbs, mutated hands, unnatural colors, noisy background, out of focus, poor composition, cultural clichés, stereotype exaggeration, flat lighting, glitch"]

    ns = model.get_noise_schedule()
    mf = model.get_model_fn(noise_schedule=ns, pos_conds=pos, neg_conds=neg, guidance_scale=4.5)
    z  = model.get_noise(seeds=[42])

    _first_pil = lambda x: (x[0] if isinstance(x, list) else x)
    def _decode(lat):
        im = _first_pil(model.decode_vae(lat, pil_output=True)['pil_output']); assert isinstance(im, Image.Image); return im
    def _outsamp(o): return o['samples'] if isinstance(o, dict) else o

    imgs, titles = [], []
    with torch.no_grad():
        # e20 = Euler_Solver(ns, steps=20, skip_type='time_uniform_flow', flow_shift=3.0, algorithm_type="data_prediction")
        # imgs.append(_decode(_outsamp(e20.sample(z.clone(), mf)))); titles.append("1. Euler 20steps")

        e5  = Euler_Solver(ns, steps=steps,  skip_type='time_uniform_flow', flow_shift=flow_shift, algorithm_type="data_prediction")
        imgs.append(_decode(_outsamp(e5.sample(z.clone(), mf)['samples'])));  titles.append("2. Euler")

        dpm = DPM_Solver(ns, algorithm_type="data_prediction", steps=steps, skip_type='time_uniform_flow', flow_shift=flow_shift)
        imgs.append(_decode(dpm.sample(z.clone(), mf)['samples'])); titles.append("3. DPM")

        gdual = GDual_Solver(ns, steps=steps, transform=LogAffineTransform(gamma_push=True, gamma_max=2, tau_offset=1, kappa_max=2, eps=1e-2),
                            param_extractor=Extractor(steps=steps), skip_type="time_uniform_flow", flow_shift=flow_shift,
                            pred_order=1, corr_order=2, order1_kappa=True, order2_kappa=True,
                            use_corrector=True, time_learning=True, train_mode=True).to(model.device)
        gdual.load_state_dict(torch.load(pt_file, map_location='cpu', weights_only=False)['solver_state_dict'], strict=True)
        imgs.append(_decode(gdual.sample(z.clone(), mf)['samples'])); titles.append("4. GDual")

    fig, axs = plt.subplots(1, 3, figsize=(24, 6), dpi=120)
    for ax, im, t in zip(axs, imgs, titles):
        ax.imshow(im); ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()


RuntimeError: Error(s) in loading state_dict for GDual_Solver:
	size mismatch for log_deltas: copying a param with shape torch.Size([6]) from checkpoint, the shape in current model is torch.Size([3]).
	size mismatch for param_extractor.table: copying a param with shape torch.Size([6, 2, 5]) from checkpoint, the shape in current model is torch.Size([3, 2, 5]).